# Generate synthetic sites (MLP and harmonization)

This notebook produces synthetic sites from harmonized subset.


In [ ]:
# Imports and configuration
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.clinical_combat.robust.synthectic_sites_generations import generate_sites



In [ ]:
init_harmonization_methods = ['pairwise']

sample_sizes = [100]
disease_ratios = [0.03, 0.1, 0.3, 0.5, 0.7, 0.8]
num_tests = 40
disease = 'ALL'            # 'ALL', or a specific disease (e.g., 'AD')

fixed_bias = True          # If False, the bias will be randomly generated for each test
centered_bias = False      # Will remove covariate effect before applying the biais
n_jobs = -1                # -1 for all cores
camcan_hc_only = True      # Filter HC outside CamCAN
include_camcan = True      # Include compilations with CamCAN
compilation_suffix = '.with_camcan' if include_camcan else ''

# Augmentation choice (None to use the original data, or an integer -> *_AUG_{n})
augmentation_copies = 5
augmentation_suffix = f"_AUG_{augmentation_copies}" if augmentation_copies else ''

def build_dataset_config(harmonization_method: str):
    input_dir = Path(f"DATA/processed/compilation/{harmonization_method}/{'harmonized'}{augmentation_suffix}")
    return {
        'name': f"{harmonization_method}{augmentation_suffix}",
        'data_path': input_dir / f"compilation.all_metrics{compilation_suffix}.csv.gz",
        'output_dir': Path(f"DATA/processed/synthetic_sites/{harmonization_method}/harmonized{augmentation_suffix}/{disease}"),
    }

datasets = [
    build_dataset_config(method)
    for method in init_harmonization_methods
]


In [ ]:
# Dataset preview (first rows)
for cfg in datasets:
    print(f"Dataset: {cfg['name']} -> {cfg['data_path']}")
    display(pd.read_csv(cfg['data_path']).head(3))
    print('-'*60)


In [ ]:
# Generate synthetic sites for each dataset
for cfg in datasets:
    cfg['output_dir'].mkdir(parents=True, exist_ok=True)
    print(f"\nGenerating for {cfg['name']} -> {cfg['output_dir']}\n")
    generate_sites(
        sample_sizes=sample_sizes,
        disease_ratios=disease_ratios,
        num_tests=num_tests,
        directory=str(cfg['output_dir']),
        data_path=str(cfg['data_path']),
        disease=disease,
        camcan_hc_only=camcan_hc_only,
        fixed_bias=fixed_bias,
        centered_bias=centered_bias,
        n_jobs=n_jobs,
        
    )
    print(f"Done: {cfg['output_dir'].resolve()}")
